# Final Capstone: MLB & Pitch Clock Analytics 

In [1]:
# import all requirements
import pandas as pd 
import requests 
import sqlalchemy
import notebook
from sqlalchemy import create_engine
from sqlalchemy import text

---
## Part 1: Data Ingestion

In this part you will load both data sources into pandas DataFrames. Nothing gets cleaned yet — just get the raw data in.

In [2]:

def get_all_player_ids(season=2025):
    
    teams_url = "https://statsapi.mlb.com/api/v1/teams"
    params = {"sportId": 1, "season": season} 
    
    response = requests.get(teams_url, params=params)
    response.raise_for_status()
    teams_data = response.json()
    
    all_players = {}
    
    for team in teams_data.get("teams", []):
        team_id = team["id"]
        team_name = team["name"]
        
        roster_url = f"https://statsapi.mlb.com/api/v1/teams/{team_id}/roster"
        roster_params = {"season": season}
        
        roster_response = requests.get(roster_url, params=roster_params)
        roster_response.raise_for_status()
        roster_data = roster_response.json()
        
        for player in roster_data.get("roster", []):
            player_id = player["person"]["id"]
            player_name = player["person"]["fullName"]
            
            if player_id not in all_players:
                all_players[player_id] = {
                    "name": player_name,
                    "team": team_name,
                    "team_id": team_id,
                    "position": player.get("position", {}).get("abbreviation", "N/A")
                }
    
    return all_players

players = get_all_player_ids(season=2025)
print(f"Found {len(players)} active players")

df_players = pd.DataFrame([
    {"player_id": player_id, **player_info} 
    for player_id, player_info in players.items()
])

print(df_players.head())

all_player_ids = df_players['player_id'].tolist()
print(f"\nAll player IDs (first 10): {all_player_ids[:10]}")

Found 1470 active players
   player_id             name       team  team_id position
0     680769     CJ Alexander  Athletics      133       1B
1     669920  Jason Alexander  Athletics      133        P
2     665660   Elvis Alvarado  Athletics      133        P
3     609280   Miguel Andujar  Athletics      133       LF
4     682183       Drew Avans  Athletics      133       LF

All player IDs (first 10): [680769, 669920, 665660, 609280, 682183, 686930, 669620, 674370, 668709, 641386]


In [8]:

def fetch_mlb_api(endpoint, **params):
    base_url = "https://statsapi.mlb.com/api/v1"
    url = f"{base_url}/{endpoint}"
    
    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

def extract_player_stats_to_dataframe(api_response):
    if 'stats' in api_response and api_response['stats']:
        stats_data = api_response['stats'][0]
        
        if 'splits' in stats_data and stats_data['splits']:
            player_stats = stats_data['splits'][0]
            
            stat_dict = player_stats.get('stat', {})
            metadata = {
                'player_id': player_stats.get('player', {}).get('id'),
                'player_name': player_stats.get('player', {}).get('fullName'),
                'season': player_stats.get('season'),
                'team_id': player_stats.get('team', {}).get('id'),
                'team_name': player_stats.get('team', {}).get('name'),
                'league_id': player_stats.get('league', {}).get('id'),
                'league_name': player_stats.get('league', {}).get('name'),
                'game_type': player_stats.get('gameType'),
                'stat_type': stats_data.get('type', {}).get('displayName'),
                'stat_group': stats_data.get('group', {}).get('displayName')
            }
            
            combined_dict = {**metadata, **stat_dict}
            
            for key, value in combined_dict.items():
                if isinstance(value, str) and value.replace('.', '').replace('-', '').isdigit():
                    if '.' in value:
                        combined_dict[key] = float(value)
                    else:
                        combined_dict[key] = int(value)
            
            df = pd.DataFrame([combined_dict])
            
            return df
    
    return pd.DataFrame()

all_data = []
years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

for year in years:
    for player in all_player_ids:
        endpoint = f"people/{player}/stats"
        params = {
            "stats": "season",
            "season": year,
            "gameType": "R" 
        }
        player = fetch_mlb_api(endpoint, **params)
        player = extract_player_stats_to_dataframe(player)
        all_data.append(player)

mlb_data = pd.concat(all_data, ignore_index=True)
print(mlb_data)

      player_id      player_name  season team_id             team_name  \
0        609280   Miguel Andujar    2019     147      New York Yankees   
1        664913       Seth Brown    2019     133     Oakland Athletics   
2        600917     José Leclerc    2019     140         Texas Rangers   
3        519008   T.J. McFarland    2019     109  Arizona Diamondbacks   
4        656794     Sean Newcomb    2019     144        Atlanta Braves   
...         ...              ...     ...     ...                   ...   
5902     663399  Brandon Waddell    2025     121         New York Mets   
5903     681810    Austin Warren    2025     121         New York Mets   
5904     608385     Jesse Winker    2025     121         New York Mets   
5905     664849      Danny Young    2025     121         New York Mets   
5906     676724      Jared Young    2025     121         New York Mets   

     league_id      league_name game_type stat_type stat_group  ...  \
0          103  American League         

---
## Part 2: Exploratory Data Analysis (EDA)

Before cleaning anything, understand what you have. For each table check:
- **Shape** — rows and columns
- **Dtypes** — are dates stored as strings? Numbers as objects?
- **Nulls** — which columns have missing values?
- **Duplicates** — any exact duplicate rows?
- **Value distributions** — unique values in key categorical columns

Useful methods: `.shape`, `.dtypes`, `.isnull().sum()`, `.duplicated().sum()`, `.value_counts()`, `.describe()`

In [25]:
print(mlb_data.shape)
print(mlb_data.dtypes)
pd.set_option('display.max_columns', None)
mlb_data.head(5)


(5907, 77)
player_id                   int64
player_name                   str
season                      int64
team_id                    object
team_name                  object
                           ...   
hitsPer9Inn                object
runsScoredPer9             object
homeRunsPer9               object
inheritedRunners          float64
inheritedRunnersScored    float64
Length: 77, dtype: object


,player_id,player_name,season,team_id,team_name,league_id,league_name,game_type,stat_type,stat_group,age,gamesPlayed,groundOuts,airOuts,runs,doubles,triples,homeRuns,strikeOuts,baseOnBalls,intentionalWalks,hits,hitByPitch,avg,atBats,obp,slg,ops,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,groundIntoDoublePlay,numberOfPitches,plateAppearances,totalBases,rbi,leftOnBase,sacBunts,sacFlies,babip,groundOutsToAirouts,catchersInterference,atBatsPerHomeRun,gamesStarted,era,inningsPitched,wins,losses,saves,saveOpportunities,holds,blownSaves,earnedRuns,whip,battersFaced,outs,gamesPitched,completeGames,shutouts,strikes,strikePercentage,hitBatsmen,balks,wildPitches,pickoffs,winPercentage,pitchesPerInning,gamesFinished,strikeoutWalkRatio,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored
0,609280,Miguel Andujar,2019,147,New York Yankees,103,American League,R,season,hitting,24.0,12,15,16,1,0,0,0,11,1,0,6,0,0.128,47,0.143,0.128,0.271,0,0,.---,.---,4,175,49.0,6,1.0,27.0,0,1,0.162,0.94,0,-.--,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,664913,Seth Brown,2019,133,Oakland Athletics,103,American League,R,season,hitting,26.0,26,13,17,11,8,2,0,23,7,0,22,1,0.293,75,0.361,0.453,0.814,0,1,1.0,0.0,2,333,83.0,34,13.0,48.0,0,0,0.423,0.76,0,-.--,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,600917,José Leclerc,2019,140,Texas Rangers,103,American League,R,season,pitching,25.0,70,40,62,34,16,2,7,100,39,1,52,6,0.209,249,0.328,0.373,0.701,0,5,1.0,0.0,4,1294,NaN,93,NaN,NaN,3,2,NaN,0.65,0,NaN,3.0,4.33,68.2,2.0,4.0,14.0,18.0,7.0,4.0,33.0,1.33,299.0,206.0,70.0,0.0,0.0,789.0,0.61,6.0,0.0,7.0,0.0,0.333,18.84,40.0,2.56,13.11,5.11,6.82,4.46,0.92,14.0,4.0
3,519008,T.J. McFarland,2019,109,Arizona Diamondbacks,104,National League,R,season,pitching,30.0,51,89,34,35,13,2,6,35,20,5,71,1,0.316,225,0.371,0.471,0.842,5,2,0.286,0.714,5,897,NaN,106,NaN,NaN,2,2,NaN,2.62,0,NaN,0.0,4.82,56.0,0.0,0.0,0.0,0.0,9.0,0.0,30.0,1.63,250.0,168.0,51.0,0.0,0.0,557.0,0.62,1.0,0.0,1.0,3.0,.---,16.02,13.0,1.75,5.63,3.21,11.41,5.63,0.96,32.0,12.0
4,656794,Sean Newcomb,2019,144,Atlanta Braves,104,National League,R,season,pitching,26.0,55,73,62,28,10,1,8,65,29,1,61,3,0.236,259,0.317,0.375,0.692,1,4,0.8,0.2,5,1181,NaN,97,NaN,NaN,0,2,NaN,1.18,0,NaN,4.0,3.16,68.1,6.0,3.0,1.0,3.0,16.0,2.0,24.0,1.32,293.0,205.0,55.0,0.0,0.0,740.0,0.63,3.0,0.0,4.0,0.0,0.667,17.28,4.0,2.24,8.56,3.82,8.03,3.69,1.05,18.0,7.0


In [13]:
# How many null values?

for columns in mlb_data:
    print(mlb_data.isnull().sum())

player_id                    0
player_name                  0
season                       0
team_id                    579
team_name                  579
                          ... 
hitsPer9Inn               2753
runsScoredPer9            2753
homeRunsPer9              2753
inheritedRunners          2753
inheritedRunnersScored    2753
Length: 77, dtype: int64
player_id                    0
player_name                  0
season                       0
team_id                    579
team_name                  579
                          ... 
hitsPer9Inn               2753
runsScoredPer9            2753
homeRunsPer9              2753
inheritedRunners          2753
inheritedRunnersScored    2753
Length: 77, dtype: int64
player_id                    0
player_name                  0
season                       0
team_id                    579
team_name                  579
                          ... 
hitsPer9Inn               2753
runsScoredPer9            2753
homeRunsPer9        

In [14]:
# Explore the MLB DataFrame

print(mlb_data.shape)


(5907, 77)


In [15]:
print(mlb_data.dtypes)

player_id                   int64
player_name                   str
season                      int64
team_id                    object
team_name                  object
                           ...   
hitsPer9Inn                object
runsScoredPer9             object
homeRunsPer9               object
inheritedRunners          float64
inheritedRunnersScored    float64
Length: 77, dtype: object


In [16]:
print(mlb_data.value_counts())

Series([], Name: count, dtype: int64)


In [17]:
print(mlb_data.duplicated().sum())

0


In [18]:
print(mlb_data.isnull().sum())

player_id                    0
player_name                  0
season                       0
team_id                    579
team_name                  579
                          ... 
hitsPer9Inn               2753
runsScoredPer9            2753
homeRunsPer9              2753
inheritedRunners          2753
inheritedRunnersScored    2753
Length: 77, dtype: int64


In [19]:
print(mlb_data.describe())

           player_id       season          age  gamesPlayed   groundOuts  \
count    5907.000000  5907.000000  5682.000000  5907.000000  5907.000000   
mean   639090.444557  2022.782631    27.449842    53.005417    61.056543   
std     53026.314324     1.916338     3.520659    46.734202    51.228446   
min    434378.000000  2019.000000    19.000000     1.000000     0.000000   
25%    608371.000000  2021.000000    25.000000    16.000000    19.000000   
50%    657656.000000  2023.000000    27.000000    34.000000    48.000000   
75%    670810.000000  2024.000000    30.000000    75.000000    94.000000   
max    829272.000000  2025.000000    45.000000   163.000000   304.000000   

           airOuts         runs      doubles      triples     homeRuns  ...  \
count  5907.000000  5907.000000  5907.000000  5907.000000  5907.000000  ...   
mean     65.373794    33.104452    12.053496     1.036059     8.783985  ...   
std      54.683083    27.838112    10.708567     1.503879     8.910534  ...   

---
## Part 3: Data Cleaning

Fix every issue you found in EDA. Only after cleaning should data be written to the database.

In [10]:
mlb_pitchers = mlb_data

In [11]:
mlb_hitters = mlb_data

In [24]:
mlb_pitchers = mlb_data.loc[mlb_data['stat_group'] == 'pitching']
mlb_pitchers.tail(5)

,player_id,player_name,season,team_id,team_name,league_id,league_name,game_type,stat_type,stat_group,age,gamesPlayed,groundOuts,airOuts,runs,doubles,triples,homeRuns,strikeOuts,baseOnBalls,intentionalWalks,hits,hitByPitch,avg,atBats,obp,slg,ops,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,groundIntoDoublePlay,numberOfPitches,plateAppearances,totalBases,rbi,leftOnBase,sacBunts,sacFlies,babip,groundOutsToAirouts,catchersInterference,atBatsPerHomeRun,gamesStarted,era,inningsPitched,wins,losses,saves,saveOpportunities,holds,blownSaves,earnedRuns,whip,battersFaced,outs,gamesPitched,completeGames,shutouts,strikes,strikePercentage,hitBatsmen,balks,wildPitches,pickoffs,winPercentage,pitchesPerInning,gamesFinished,strikeoutWalkRatio,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored
5898,694918,Blade Tidwell,2025,121,New York Mets,104,National League,R,season,pitching,24.0,4,15,19,15,2,0,4,10,10,0,23,1,0.348,66,0.436,0.561,0.997,0,2,1.0,0.0,1,309,NaN,37,NaN,NaN,0,1,NaN,0.79,0,NaN,2.0,9.0,15.0,1.0,1.0,0.0,0.0,0.0,0.0,15.0,2.2,78.0,45.0,4.0,0.0,0.0,190.0,0.61,1.0,0.0,0.0,0.0,0.5,20.6,1.0,1.0,6.0,6.0,13.8,9.0,2.4,1.0,0.0
5899,804636,Jonah Tong,2025,121,New York Mets,104,National League,R,season,pitching,22.0,5,12,20,20,5,0,3,22,9,0,24,0,0.312,77,0.379,0.494,0.873,1,0,0.0,1.0,1,371,NaN,38,NaN,NaN,0,1,NaN,0.6,0,NaN,5.0,7.71,18.2,2.0,3.0,0.0,0.0,0.0,0.0,16.0,1.77,87.0,56.0,5.0,0.0,0.0,236.0,0.64,0.0,0.0,3.0,0.0,0.4,19.88,0.0,2.44,10.61,4.34,11.57,9.64,1.45,0.0,0.0
5902,663399,Brandon Waddell,2025,121,New York Mets,104,National League,R,season,pitching,31.0,11,32,39,12,10,1,4,22,11,0,29,1,0.240,121,0.306,0.438,0.744,0,1,1.0,0.0,2,512,NaN,53,NaN,NaN,0,1,NaN,0.82,0,NaN,1.0,3.45,31.1,0.0,0.0,0.0,0.0,1.0,0.0,12.0,1.28,134.0,94.0,11.0,0.0,0.0,315.0,0.62,1.0,0.0,0.0,0.0,.---,16.34,4.0,2.0,6.32,3.16,8.33,3.45,1.15,7.0,1.0
5903,681810,Austin Warren,2025,121,New York Mets,104,National League,R,season,pitching,29.0,5,7,9,1,0,0,1,9,4,0,5,0,0.167,30,0.265,0.267,0.532,0,0,.---,.---,2,146,NaN,8,NaN,NaN,0,0,NaN,0.78,0,NaN,0.0,0.96,9.1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.96,34.0,28.0,5.0,0.0,0.0,86.0,0.59,0.0,0.0,0.0,0.0,1.0,15.64,1.0,2.25,8.68,3.86,4.82,0.96,0.96,6.0,1.0
5905,664849,Danny Young,2025,121,New York Mets,104,National League,R,season,pitching,31.0,10,9,2,5,0,0,0,13,3,0,9,1,0.281,32,0.361,0.281,0.642,0,0,.---,.---,1,158,NaN,9,NaN,NaN,1,0,NaN,4.5,0,NaN,0.0,4.32,8.1,0.0,0.0,0.0,0.0,4.0,0.0,4.0,1.44,37.0,25.0,10.0,0.0,0.0,91.0,0.58,1.0,0.0,0.0,0.0,.---,18.96,4.0,4.33,14.04,3.24,9.72,5.4,0.0,1.0,0.0


In [27]:
mlb_pitchers = mlb_pitchers.dropna(axis=1, how='all')
mlb_pitchers.tail(5)

,player_id,player_name,season,team_id,team_name,league_id,league_name,game_type,stat_type,stat_group,age,gamesPlayed,groundOuts,airOuts,runs,doubles,triples,homeRuns,strikeOuts,baseOnBalls,intentionalWalks,hits,hitByPitch,avg,atBats,obp,slg,ops,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,groundIntoDoublePlay,numberOfPitches,totalBases,sacBunts,sacFlies,groundOutsToAirouts,catchersInterference,gamesStarted,era,inningsPitched,wins,losses,saves,saveOpportunities,holds,blownSaves,earnedRuns,whip,battersFaced,outs,gamesPitched,completeGames,shutouts,strikes,strikePercentage,hitBatsmen,balks,wildPitches,pickoffs,winPercentage,pitchesPerInning,gamesFinished,strikeoutWalkRatio,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored
5898,694918,Blade Tidwell,2025,121,New York Mets,104,National League,R,season,pitching,24.0,4,15,19,15,2,0,4,10,10,0,23,1,0.348,66,0.436,0.561,0.997,0,2,1.0,0.0,1,309,37,0,1,0.79,0,2.0,9.0,15.0,1.0,1.0,0.0,0.0,0.0,0.0,15.0,2.2,78.0,45.0,4.0,0.0,0.0,190.0,0.61,1.0,0.0,0.0,0.0,0.5,20.6,1.0,1.0,6.0,6.0,13.8,9.0,2.4,1.0,0.0
5899,804636,Jonah Tong,2025,121,New York Mets,104,National League,R,season,pitching,22.0,5,12,20,20,5,0,3,22,9,0,24,0,0.312,77,0.379,0.494,0.873,1,0,0.0,1.0,1,371,38,0,1,0.6,0,5.0,7.71,18.2,2.0,3.0,0.0,0.0,0.0,0.0,16.0,1.77,87.0,56.0,5.0,0.0,0.0,236.0,0.64,0.0,0.0,3.0,0.0,0.4,19.88,0.0,2.44,10.61,4.34,11.57,9.64,1.45,0.0,0.0
5902,663399,Brandon Waddell,2025,121,New York Mets,104,National League,R,season,pitching,31.0,11,32,39,12,10,1,4,22,11,0,29,1,0.240,121,0.306,0.438,0.744,0,1,1.0,0.0,2,512,53,0,1,0.82,0,1.0,3.45,31.1,0.0,0.0,0.0,0.0,1.0,0.0,12.0,1.28,134.0,94.0,11.0,0.0,0.0,315.0,0.62,1.0,0.0,0.0,0.0,.---,16.34,4.0,2.0,6.32,3.16,8.33,3.45,1.15,7.0,1.0
5903,681810,Austin Warren,2025,121,New York Mets,104,National League,R,season,pitching,29.0,5,7,9,1,0,0,1,9,4,0,5,0,0.167,30,0.265,0.267,0.532,0,0,.---,.---,2,146,8,0,0,0.78,0,0.0,0.96,9.1,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.96,34.0,28.0,5.0,0.0,0.0,86.0,0.59,0.0,0.0,0.0,0.0,1.0,15.64,1.0,2.25,8.68,3.86,4.82,0.96,0.96,6.0,1.0
5905,664849,Danny Young,2025,121,New York Mets,104,National League,R,season,pitching,31.0,10,9,2,5,0,0,0,13,3,0,9,1,0.281,32,0.361,0.281,0.642,0,0,.---,.---,1,158,9,1,0,4.5,0,0.0,4.32,8.1,0.0,0.0,0.0,0.0,4.0,0.0,4.0,1.44,37.0,25.0,10.0,0.0,0.0,91.0,0.58,1.0,0.0,0.0,0.0,.---,18.96,4.0,4.33,14.04,3.24,9.72,5.4,0.0,1.0,0.0


In [22]:
mlb_hitters = mlb_data.loc[mlb_data['stat_group'] == 'hitting']
mlb_hitters.tail(5)

,player_id,player_name,season,team_id,team_name,league_id,league_name,game_type,stat_type,stat_group,age,gamesPlayed,groundOuts,airOuts,runs,doubles,triples,homeRuns,strikeOuts,baseOnBalls,intentionalWalks,hits,hitByPitch,avg,atBats,obp,slg,ops,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,groundIntoDoublePlay,numberOfPitches,plateAppearances,totalBases,rbi,leftOnBase,sacBunts,sacFlies,babip,groundOutsToAirouts,catchersInterference,atBatsPerHomeRun,gamesStarted,era,inningsPitched,wins,losses,saves,saveOpportunities,holds,blownSaves,earnedRuns,whip,battersFaced,outs,gamesPitched,completeGames,shutouts,strikes,strikePercentage,hitBatsmen,balks,wildPitches,pickoffs,winPercentage,pitchesPerInning,gamesFinished,strikeoutWalkRatio,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored
5897,621438,Tyrone Taylor,2025,121,New York Mets,104,National League,R,season,hitting,31.0,113,63,107,34,18,3,2,76,16,1,69,9,0.223,310,0.279,0.319,0.598,2,12,0.857,0.143,4,1246,341.0,99,27.0,130.0,3,2,0.286,0.59,1,155.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5900,620443,Luis Torrens,2025,121,New York Mets,104,National League,R,season,hitting,29.0,93,89,58,20,14,1,5,56,19,0,59,2,0.226,261,0.284,0.345,0.629,0,1,1.0,0.0,7,1069,283.0,90,29.0,100.0,1,0,0.27,1.53,0,52.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5901,668901,Mark Vientos,2025,121,New York Mets,104,National League,R,season,hitting,25.0,121,113,101,44,21,2,17,115,30,0,99,5,0.233,424,0.289,0.413,0.702,0,1,1.0,0.0,11,1764,463.0,175,61.0,188.0,0,4,0.277,1.12,0,24.94,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5904,608385,Jesse Winker,2025,121,New York Mets,104,National League,R,season,hitting,31.0,26,14,21,8,5,2,1,21,9,1,16,0,0.229,70,0.309,0.400,0.709,0,1,1.0,0.0,1,317,81.0,28,10.0,37.0,0,2,0.3,0.67,0,70.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5906,676724,Jared Young,2025,121,New York Mets,104,National League,R,season,hitting,29.0,23,6,14,5,1,0,4,16,2,0,8,1,0.186,43,0.234,0.488,0.722,0,0,.---,.---,1,192,47.0,21,6.0,15.0,0,1,0.167,0.43,0,10.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
mlb_hitters = mlb_hitters.dropna(axis=1, how='all')
mlb_hitters.tail(5)

,player_id,player_name,season,team_id,team_name,league_id,league_name,game_type,stat_type,stat_group,age,gamesPlayed,groundOuts,airOuts,runs,doubles,triples,homeRuns,strikeOuts,baseOnBalls,intentionalWalks,hits,hitByPitch,avg,atBats,obp,slg,ops,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,groundIntoDoublePlay,numberOfPitches,plateAppearances,totalBases,rbi,leftOnBase,sacBunts,sacFlies,babip,groundOutsToAirouts,catchersInterference,atBatsPerHomeRun
5897,621438,Tyrone Taylor,2025,121,New York Mets,104,National League,R,season,hitting,31.0,113,63,107,34,18,3,2,76,16,1,69,9,0.223,310,0.279,0.319,0.598,2,12,0.857,0.143,4,1246,341.0,99,27.0,130.0,3,2,0.286,0.59,1,155.0
5900,620443,Luis Torrens,2025,121,New York Mets,104,National League,R,season,hitting,29.0,93,89,58,20,14,1,5,56,19,0,59,2,0.226,261,0.284,0.345,0.629,0,1,1.0,0.0,7,1069,283.0,90,29.0,100.0,1,0,0.27,1.53,0,52.2
5901,668901,Mark Vientos,2025,121,New York Mets,104,National League,R,season,hitting,25.0,121,113,101,44,21,2,17,115,30,0,99,5,0.233,424,0.289,0.413,0.702,0,1,1.0,0.0,11,1764,463.0,175,61.0,188.0,0,4,0.277,1.12,0,24.94
5904,608385,Jesse Winker,2025,121,New York Mets,104,National League,R,season,hitting,31.0,26,14,21,8,5,2,1,21,9,1,16,0,0.229,70,0.309,0.400,0.709,0,1,1.0,0.0,1,317,81.0,28,10.0,37.0,0,2,0.3,0.67,0,70.0
5906,676724,Jared Young,2025,121,New York Mets,104,National League,R,season,hitting,29.0,23,6,14,5,1,0,4,16,2,0,8,1,0.186,43,0.234,0.488,0.722,0,0,.---,.---,1,192,47.0,21,6.0,15.0,0,1,0.167,0.43,0,10.75
